In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import dtale
import gc

bureau_df = pd.read_parquet(cfg.CLEANS_DIR / "bureau.train-cleaned.parquet")


In [ ]:
bureau_balance_agg= pd.read_parquet(cfg.PROCESSED_DIR / "bureu_balance_agg.parquet")
bureau_df=bureau_df.merge(bureau_balance_agg,how="left",on="id_bureau")
bureau_df['has_bureau_balance_data'] = bureau_df['balance_months_balance_min'].notna().astype(int)
bureau_balance_agg.head()


,balance_raw_length,balance_amount_rows_with_activity,balance_potential_on_going_loan,balance_incomplete_sequence,balance_closing_month,balance_months_balance_min,balance_months_balance_max,balance_status_score_max,balance_status_score_mean,balance_status_score_std,...,balance_status_3_mean,balance_status_3_sum,balance_status_4_mean,balance_status_4_sum,balance_status_5_mean,balance_status_5_sum,balance_status_c_mean,balance_status_c_sum,balance_status_x_mean,balance_status_x_sum
id_bureau,,,,,,,,,,,,,,,,,,,,,
5008804,16,3,False,False,-12.0,-15,0,1,0.062500,0.250000,...,0.0,0,0.0,0,0.0,0,0.812500,13,0.062500,1
5008805,15,3,False,False,-11.0,-14,0,1,0.066667,0.258199,...,0.0,0,0.0,0,0.0,0,0.800000,12,0.066667,1
5008806,30,23,False,False,-6.0,-29,0,0,0.000000,0.000000,...,0.0,0,0.0,0,0.0,0,0.233333,7,0.533333,16
5008807,8,9,False,True,NaN,-87,-80,0,0.000000,0.000000,...,0.0,0,0.0,0,0.0,0,0.000000,0,0.125000,1
5008808,5,6,True,False,NaN,-4,0,0,0.000000,0.000000,...,0.0,0,0.0,0,0.0,0,0.000000,0,0.600000,3


In [12]:
del bureau_balance_agg
gc.collect()

1825

In [ ]:
bureau_agg_dic= {
    "id_curr": ["count"],

    #monetary
    "amt_credit_sum": ["max", "min","sum"],
    "amt_credit_sum_limit": ["max","mean","min"],
    "amt_annuity" : ["max","mean","min"],
    "amt_credit_sum_debt" : ["max","mean","sum"],


    #log_transformed
    "log_amt_credit_sum": ["mean","std"],

    #counters
    "cnt_credit_prolong": ["max","mean","sum"],
    "days_credit_update": ["min","max","mean"], 
    "days_credit": ["min","max","mean"], 
    "days_enddate_fact": ["max"], 

    #other
    "ratio_credit_annuity" : ["max","mean","min"],
    
    #categoricals
    "credit_active_active" : ["mean","sum"],
    "credit_active_closed" : ["mean","sum"],
    "credit_active_sold" : ["mean","sum"],
    "amt_credit_sum_overdue_is_missing": ["mean","sum"],
    "amt_credit_sum_debt_is_negative": ["mean","sum"],
    "days_enddate_fact_is_missing" : ["mean","sum"],
    "flag_have_credit_day_overdue": ["mean","sum"],
    "have_amt_credit_sum_overdue" : ["mean","sum"],
    "amt_credit_sum_limit_is_missing" : ["mean","sum"],
    "amt_credit_sum_limit_is_zero" : ["mean","sum"],
    "amt_annuity_is_missing" : ["mean","sum"],
}

In [ ]:
agg_from_bureau_balance_dict = {
    "balance_potential_on_going_loan": ["sum"], 
    
    "balance_status_score_max": ["max"], 
    
    "balance_closing_month": ["max"],
    
    "balance_months_balance_min" : ["min"]
}

In [ ]:
bureau_df["log_amt_credit_sum"] = np.log1p(bureau_df["amt_credit_sum"])
bureau_df["ratio_credit_annuity"]= np.where(bureau_df["amt_annuity"] !=0, bureau_df["amt_credit_sum"] / bureau_df["amt_annuity"] , np.nan )  


bureau_df["credit_active"]=bureau_df["credit_active"].str.lower()
bureau_df= pd.get_dummies(bureau_df, columns=["credit_active"], dtype=int)

bureau_df.sort_values(["id_curr", "days_credit"],inplace=True,ascending=False)
last_two = bureau_df.groupby("id_curr").head(2)
last_two = last_two.copy()
last_two["loan_order"] = last_two.groupby("id_curr").cumcount() + 1
last_two_columns = last_two.pivot(index="id_curr", columns="loan_order")


bureau_aggregated = bureau_df.groupby("id_curr").agg(bureau_agg_dic | agg_from_bureau_balance_dict)

bureau_aggregated.columns= [f"{col[0]}_{col[1]}" for col in bureau_aggregated.columns]
bureau_aggregated= bureau_aggregated.reset_index()

last_two_columns.columns = [f"bureau_{col[0]}_loan_{col[1]}" for col in last_two_columns.columns]
last_two_columns= last_two_columns.reset_index()


last_two_columns.head()





,id_curr,bureau_id_bureau_loan_1,bureau_id_bureau_loan_2,bureau_credit_currency_loan_1,bureau_credit_currency_loan_2,bureau_days_credit_loan_1,bureau_days_credit_loan_2,bureau_flag_have_credit_day_overdue_loan_1,bureau_flag_have_credit_day_overdue_loan_2,bureau_days_credit_enddate_loan_1,...,bureau_log_amt_credit_sum_loan_1,bureau_log_amt_credit_sum_loan_2,bureau_ratio_credit_annuity_loan_1,bureau_ratio_credit_annuity_loan_2,bureau_credit_active_active_loan_1,bureau_credit_active_active_loan_2,bureau_credit_active_closed_loan_1,bureau_credit_active_closed_loan_2,bureau_credit_active_sold_loan_1,bureau_credit_active_sold_loan_2
0,100002,6158909.0,6158905.0,currency 1,currency 1,-103.0,-476.0,0.0,0.0,NaN,...,10.373165,0.000000,NaN,NaN,1.0,0.0,0.0,1.0,0.0,0.0
1,100003,5885880.0,5885879.0,currency 1,currency 1,-606.0,-775.0,0.0,0.0,1216.0,...,13.604791,11.193457,NaN,NaN,1.0,0.0,0.0,1.0,0.0,0.0
2,100004,6829134.0,6829133.0,currency 1,currency 1,-408.0,-1326.0,0.0,0.0,-382.0,...,11.456766,11.456366,NaN,NaN,0.0,0.0,1.0,1.0,0.0,0.0
3,100007,5987200.0,NaN,currency 1,NaN,-1149.0,NaN,0.0,NaN,-783.0,...,11.893080,NaN,NaN,NaN,0.0,NaN,1.0,NaN,0.0,NaN
4,100008,6491434.0,6491433.0,currency 1,currency 1,-78.0,-1097.0,0.0,0.0,471.0,...,12.497275,11.568417,NaN,NaN,1.0,0.0,0.0,1.0,0.0,0.0


In [ ]:
bureau_final_df= last_two_columns.merge(bureau_aggregated,how="left",on="id_curr")
bureau_final_df.head()

,id_curr,bureau_id_bureau_loan_1,bureau_id_bureau_loan_2,bureau_credit_currency_loan_1,bureau_credit_currency_loan_2,bureau_days_credit_loan_1,bureau_days_credit_loan_2,bureau_flag_have_credit_day_overdue_loan_1,bureau_flag_have_credit_day_overdue_loan_2,bureau_days_credit_enddate_loan_1,...,balance_closing_month_max,balance_months_balance_min_min,balance_status_score_max_max,balance_status_score_max_mean,balance_status_score_max_std,balance_status_1_sum_sum,balance_status_2_sum_sum,balance_status_3_sum_sum,balance_status_4_sum_sum,balance_status_5_sum_sum
0,100002,6158909.0,6158905.0,currency 1,currency 1,-103.0,-476.0,0.0,0.0,NaN,...,-12.0,-47.0,1.0,0.75,0.46291,27.0,0.0,0.0,0.0,0.0
1,100003,5885880.0,5885879.0,currency 1,currency 1,-606.0,-775.0,0.0,0.0,1216.0,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
2,100004,6829134.0,6829133.0,currency 1,currency 1,-408.0,-1326.0,0.0,0.0,-382.0,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
3,100007,5987200.0,NaN,currency 1,NaN,-1149.0,NaN,0.0,NaN,-783.0,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
4,100008,6491434.0,6491433.0,currency 1,currency 1,-78.0,-1097.0,0.0,0.0,471.0,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0


In [2]:
bureau_df = pd.read_parquet(cfg.CLEANS_DIR / "bureau.train-cleaned.parquet")

bureau_balance_agg= pd.read_parquet(cfg.PROCESSED_DIR / "bureu_balance_agg.parquet")
bureau_df=bureau_df.merge(bureau_balance_agg,how="left",on="id_bureau")
bureau_df['has_bureau_balance_data'] = bureau_df['balance_months_balance_min'].notna().astype(int)
bureau_balance_agg.head()


bureau_df["ratio_credit_annuity"]= np.where(bureau_df["amt_annuity"] !=0, bureau_df["amt_credit_sum"] / bureau_df["amt_annuity"] , np.nan )  
bureau_df["completetitud_ratio"] = np.where(bureau_df["amt_credit_sum_debt"]!=0,bureau_df["amt_credit_sum"] / bureau_df["amt_credit_sum_debt"],np.nan)
bureau_df["ratio_debt_limit"] =  np.where(bureau_df["amt_credit_sum"] !=0, bureau_df["amt_credit_sum_limit"] / bureau_df["amt_credit_sum"], np.nan )  



#bureau_df["expected_vs_factical_endate"] = (bureau_df["days_credit_enddate"] * -1) - ( bureau_df["days_enddate_fact"] * -1)


bureau_df["credit_active"]=bureau_df["credit_active"].str.lower()
#bureau_df["credit_type"]=bureau_df["credit_type"].str.lower()
bureau_df= pd.get_dummies(bureau_df, columns=["credit_active"], dtype=int)
#bureau_df= pd.get_dummies(bureau_df, columns=["credit_type"], dtype=int)


bureau_df.sort_values(["id_curr", "days_credit"],inplace=True,ascending=False)
last_two = bureau_df.groupby("id_curr").head(2)
last_two = last_two.copy()
last_two["loan_order"] = last_two.groupby("id_curr").cumcount() + 1


last_two_columns = last_two.drop(columns=["id_bureau"]).pivot(index="id_curr", columns="loan_order")
last_two_columns.columns = [f"bureau_{col[0]}_loan_{col[1]}" for col in last_two_columns.columns]
last_two_columns= last_two_columns.reset_index()

bureau_df["log_amt_credit_sum"] = np.log1p(bureau_df["amt_credit_sum"])

bureau_agg_dic= {
    "id_curr": ["count"],

    #monetary
    "amt_credit_sum": ["max", "mean","sum","std"],
    "amt_credit_sum_limit": ["max","mean","min","std"],
    "amt_annuity" : ["max","mean","min","std"],
    "amt_credit_sum_debt" : ["max","mean","sum","std"],



    #log_transformed
    "log_amt_credit_sum": ["mean","std"],

    #counters
    "cnt_credit_prolong": ["max","mean","sum"],
    "days_credit_update": ["min","max","mean"], 
    "days_credit": ["min","max","mean"], 
    "days_enddate_fact": ["max"], 
   # "expected_vs_factical_endate": ["max"],

    #other
    "ratio_credit_annuity" : ["max","mean","min"],
    "completetitud_ratio" : ["mean","min"],
    
    #categoricals
    "has_bureau_balance_data": ["sum", "mean"],
    "credit_active_active" : ["mean","sum"],
    "credit_active_closed" : ["mean","sum"],
    "credit_active_sold" : ["mean","sum"],
    #"amt_credit_sum_overdue_is_missing": ["mean","sum"],
    "amt_credit_sum_debt_is_negative": ["mean","sum"],
    "days_enddate_fact_is_missing" : ["mean","sum"],
    ##"flag_have_credit_day_overdue": ["mean","sum"],
    "have_amt_credit_sum_overdue" : ["mean","sum"],
    "amt_credit_sum_limit_is_missing" : ["mean","sum"],
    "amt_credit_sum_limit_is_zero" : ["mean","sum"],
    "amt_annuity_is_missing" : ["mean","sum"],
}

agg_from_bureau_balance_dict = {
    ###"balance_potential_on_going_loan": ["sum"], 
    "balance_status_score_max": ["max"], #0.76 
    "balance_months_balance_max": ["max"], #0.76 
    "balance_months_balance_min": ["min"], #0.76 
    "balance_months_since_delincuency" : ["max"],#0.76 
    "balance_is_delincuency_sum" : ["max"],#0.76 
    "balance_is_delincuency_mean" : ["mean"],#0.76 
    ##"balance_status_0_sum" : ["sum"],
    ##"balance_is_delincuency_sum": ["sum"]
   # "balance_months_since_2_status" : ["max"],
   # "balance_months_since_3_status" : ["max"],
   # "balance_months_since_4_status" : ["max"],
   # "balance_months_since_5_status" : ["max"],    
}


bureau_aggregated = bureau_df.groupby("id_curr").agg(bureau_agg_dic | agg_from_bureau_balance_dict)
#bureau_aggregated["balance_ratio_zero_vs_delincuency"]= np.where(bureau_aggregated["balance_is_delincuency_sum"] != 0, bureau_aggregated["balance_status_0_sum"] / bureau_aggregated["balance_is_delincuency_sum"] , np.nan ) 

bureau_aggregated.columns= [f"{col[0]}_{col[1]}" for col in bureau_aggregated.columns]
bureau_aggregated= bureau_aggregated.reset_index()

#bureau_aggregated["active_closed_diff"] =bureau_aggregated["credit_active_active_sum"] - bureau_aggregated["credit_active_closed_sum"]


bureau_final_df= last_two_columns.merge(bureau_aggregated,how="left",on="id_curr")
bureau_final_df.head()


#bureau_final_df["balance_potential_on_going_loan_sum"] = bureau_final_df["balance_potential_on_going_loan_sum"].astype("float32")

bureau_final_df.to_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

In [4]:
dtale.show(bureau_final_df)

2026-06-16 16:19:05,223 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2313, in preprocess_request
    rv = self.ensure_sync(before_func)()
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\defa

2026-06-16 16:25:10,082 - ERROR    - Exception occurred while processing request: object of type 'NoneType' has no len()
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 120, in _handle_exceptions
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 1573, in get_processes
    [_load_process(data_id) for data_id in global_state.keys()],
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 1573, in <listcomp>
    [_load_process(data_id) for data_id in global_state.keys()],
     ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 1558, in _load_process
    rows=len(data),
       

In [7]:
bureau_balance_agg= pd.read_parquet(cfg.PROCESSED_DIR / "bureu_balance_agg.parquet")
bureau_balance_agg

,id_bureau,balance_raw_length,balance_amount_rows_with_activity,balance_potential_on_going_loan,balance_incomplete_sequence,balance_closing_month,balance_months_since_1_status,balance_months_since_2_status,balance_months_since_3_status,balance_months_since_4_status,...,balance_status_3_mean,balance_status_3_sum,balance_status_4_mean,balance_status_4_sum,balance_status_5_mean,balance_status_5_sum,balance_status_c_mean,balance_status_c_sum,balance_status_x_mean,balance_status_x_sum
0,5008804,16,3,False,False,-12.0,-13.0,NaN,NaN,NaN,...,0.0,0,0.0,0,0.0,0,0.812500,13,0.062500,1
1,5008805,15,3,False,False,-11.0,-12.0,NaN,NaN,NaN,...,0.0,0,0.0,0,0.0,0,0.800000,12,0.066667,1
2,5008806,30,23,False,False,-6.0,NaN,NaN,NaN,NaN,...,0.0,0,0.0,0,0.0,0,0.233333,7,0.533333,16
3,5008807,8,9,False,True,NaN,NaN,NaN,NaN,NaN,...,0.0,0,0.0,0,0.0,0,0.000000,0,0.125000,1
4,5008808,5,6,True,False,NaN,NaN,NaN,NaN,NaN,...,0.0,0,0.0,0,0.0,0,0.000000,0,0.600000,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523510,6842884,48,28,False,False,-19.0,NaN,NaN,NaN,NaN,...,0.0,0,0.0,0,0.0,0,0.416667,20,0.395833,19
523511,6842885,24,25,True,False,NaN,NaN,NaN,NaN,NaN,...,0.0,0,0.0,0,0.5,12,0.000000,0,0.000000,0
523512,6842886,33,8,False,False,-24.0,NaN,NaN,NaN,NaN,...,0.0,0,0.0,0,0.0,0,0.757576,25,0.000000,0
523513,6842887,37,6,False,False,-30.0,NaN,NaN,NaN,NaN,...,0.0,0,0.0,0,0.0,0,0.837838,31,0.000000,0
